In [1]:
%load_ext autoreload
# autoreload 2：改了 benchmark/ 下的文件不用重启 kernel
%autoreload 2

import os
os.chdir(os.path.expanduser("~/projects/LLM/hw2"))   # benchmark 包以 hw2/ 为根

import torch
from benchmark import attention_sweep, bench_attention
from cs336_basics.model import scaled_dot_product_attention as sdpa   # §4.1 的被测对象

torch.backends.cuda.matmul.allow_tf32 = False        # fp32 基准：不偷偷走 tensor core
torch.backends.cudnn.allow_tf32 = False

PROF = "profiles"                       # nsys 报告、显存快照（gitignored）
def assets(sec):                        # 每节的落盘目录 notes/assets/s<sec>；各节开头 OUT = assets(n)
    d = f"notes/assets/s{sec}"; os.makedirs(d, exist_ok=True); return d

print(torch.cuda.get_device_name(0), torch.__version__)

NVIDIA GeForce RTX 5090 2.11.0+cu130


## 4 GPU Kernels

PDF §4。编号按 problem 顺序（4.1 `pytorch_attention` = PDF 4.1.1，4.2 `torch_compile` = PDF 4.2，
4.3–4.5 三题在 PDF 里都归在 4.2.2 下）。

In [2]:
OUT = assets(4)

### 4.1 `pytorch_attention`

被测的是 hw1 的 `scaled_dot_product_attention(q, k, v, mask=None)`，
输入 `[batch, seq, d]` —— batch 固定 8，**去掉 head 维**（题面 (i)）。

每个 (d, seq) 格子量三个数：100 次前向 ms、反向开始前的 `memory_allocated()`、100 次反向 ms；
OOM 的格子记 `status`，不中断 sweep。

In [3]:
# 全量 4 × 5 = 20 格。大格子预期 OOM —— 那正是题目要的答案
df = attention_sweep(sdpa, f"{OUT}/attention_fp32.md")
df

[20/20] d=128 seq=16384


,d,seq,fwd_ms,bwd_ms,mem_before_bwd_gib,status
0,16,256,0.057794,0.202094,0.020287,ok
1,16,1024,0.163510,0.492480,0.080414,ok
2,16,4096,4.700095,11.811696,1.024048,ok
3,16,8192,18.954626,46.520513,4.032227,ok
4,16,16384,NaN,NaN,NaN,OOM@forward
5,32,256,0.059714,0.219097,0.020775,ok
6,32,1024,0.176219,0.512858,0.082367,ok
7,32,4096,4.753815,12.024030,1.031860,ok
8,32,8192,19.040795,46.579400,4.047852,ok
9,32,16384,NaN,NaN,NaN,OOM@forward


In [ ]:
# 实验网格（题面 (ii)）：4 × 5 = 20 格。大格子预期 OOM —— 那正是题目要的答案
D_MODELS = [16, 32, 64, 128]
SEQ_LENS = [256, 1024, 4096, 8192, 16384]

df = attention_sweep(sdpa, D_MODELS, SEQ_LENS, f"{OUT}/attention_fp32.md")
df

In [4]:
import pandas as pd
MiB = 2**20
B = 8

def account(d, seq):
    qkvo = 4 * B * seq * d * 4            # Q K V O，随 seq 线性
    sp   = 2 * B * seq * seq * 4          # S=QKᵀ/√d 与 P=softmax(S)，随 seq²
    return dict(d=d, seq=seq,
                QKVO_MiB=qkvo/MiB, S_P_MiB=sp/MiB, total_GiB=(qkvo+sp)/2**30)

acc = pd.DataFrame([account(16, s) for s in SEQ_LENS])
acc["实测_GiB"] = list(df[df.d == 16].mem_before_bwd_gib)
acc

,d,seq,QKVO_MiB,S_P_MiB,total_GiB,实测_GiB
0,16,256,0.5,4.0,0.004395,0.020287
1,16,1024,2.0,64.0,0.064453,0.080414
2,16,4096,8.0,1024.0,1.007812,1.024048
3,16,8192,16.0,4096.0,4.015625,4.032227
4,16,16384,32.0,16384.0,16.031250,NaN
